# Week 10 Lab — Machine Learning & Dashboards

**Student:** Niyitanga Denys  
**Student ID:** 29125  
**Program:** Software Engineering  
**Course:** Introduction to Big Data Analytics

## Setup and load the dataset
The dataset CSV is included alongside this notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, confusion_matrix

# Load the supplied student dataset
df = pd.read_csv("week10_students.csv")
df.head()

,student_id,name,gender,program,district,attendance_pct,quiz_avg,assignments_avg,score
0,AUCA001,Aline Uwase,F,Information Systems,Gasabo,55,31.0,35.0,28.6
1,AUCA002,Eric Iradukunda,M,Software Engineering,Musanze,73,61.1,76.7,64.5
2,AUCA003,Diane Uwizeyimana,F,Networking,Muhanga,92,35.2,65.5,52.7
3,AUCA004,Jean Bosco Ikirezi,M,Accounting,Kicukiro,69,65.2,54.2,68.3
4,AUCA005,Patrick Ingabire,M,Information Systems,Huye,88,39.3,42.9,42.3


## Part 1 — Explore and split

### Exercise 1.1 — Know your data

In [2]:
print("Dataset shape:", df.shape)
correlations = df[["quiz_avg", "assignments_avg", "attendance_pct", "score"]].corr()["score"].drop("score")
print("Correlation with score:")
print(correlations)
# The feature with the largest absolute correlation is the strongest simple linear indicator of score.

Dataset shape: (150, 9)
Correlation with score:
quiz_avg           0.800726
assignments_avg    0.535983
attendance_pct     0.238118
Name: score, dtype: float64


### Exercise 1.2 — Frame and split

In [3]:
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
X = df[features]
y = df["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print("Training set:", X_train.shape[0], "students")
print("Test set:", X_test.shape[0], "students")
# Holding out 25% gives an independent group for checking how well the model generalizes to unseen students.

Training set: 112 students
Test set: 38 students


## Part 2 — Regression: predict the score

### Exercise 2.1 — Train and read the model

In [4]:
reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
for feature, coefficient in zip(features, reg_model.coef_):
    print(f"{feature}: {coefficient:.4f}")
print("Intercept:", round(reg_model.intercept_, 4))
# The quiz coefficient estimates the score-point change associated with one additional quiz point,
# while the other two input features are held constant.

quiz_avg: 0.5790
assignments_avg: 0.3648
attendance_pct: 0.1134
Intercept: -4.4282


### Exercise 2.2 — Judge it honestly

In [ ]:
y_pred = reg_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Test MAE: {mae:.3f}")
print(f"Test R²: {r2:.3f}")
# MAE is the average absolute prediction error in score points. 
# R² is the proportion of score variation explained by the model on students not used for training.

Test MAE: 3.329
Test R²: 0.887


### Exercise 2.3 — Predict new students

In [6]:
new_student = pd.DataFrame([{
    "quiz_avg": 70,
    "assignments_avg": 75,
    "attendance_pct": 90
}])
print("Prediction for the provided new student:", round(reg_model.predict(new_student)[0], 2))

# Illustrative personal input values (invented for this exercise, not actual academic records).
my_student = pd.DataFrame([{
    "quiz_avg": 80,
    "assignments_avg": 85,
    "attendance_pct": 90
}])
print("Prediction using illustrative personal values:", round(reg_model.predict(my_student)[0], 2))

Prediction for the provided new student: 73.66
Prediction using illustrative personal values: 83.1


## Part 3 — Classification: catch at-risk students

### Exercise 3.1 — Build the target and train

In [7]:
df["passed"] = (df["score"] >= 50).astype(int)
print("Pass/fail counts (0 = fail, 1 = pass):")
print(df["passed"].value_counts().sort_index())

X_class = df[features]
y_class = df["passed"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_class, y_class, test_size=0.25, random_state=42
)
classifier = LogisticRegression(max_iter=1000)
classifier.fit(Xc_train, yc_train)

Pass/fail counts (0 = fail, 1 = pass):
passed
0     19
1    131
Name: count, dtype: int64


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

### Exercise 3.2 — The accuracy trap

In [8]:
yc_pred = classifier.predict(Xc_test)
model_accuracy = accuracy_score(yc_test, yc_pred)
lazy_baseline_accuracy = yc_test.mean()  # Predicting pass (1) for every test student
cm = confusion_matrix(yc_test, yc_pred, labels=[0, 1])
print(f"Logistic regression accuracy: {model_accuracy:.3f}")
print(f"Always-pass baseline accuracy: {lazy_baseline_accuracy:.3f}")
print("Confusion matrix (rows = actual [fail, pass], columns = predicted [fail, pass]):")
print(cm)
# Similar overall accuracies do not automatically make the model useless. The confusion matrix
# reveals which individual cases are correctly or incorrectly identified. In particular, inspect
# the fail row: correctly identifying at-risk students (true negatives for class 0) matters for
# targeted support, while false predictions can still misdirect support. Accuracy alone hides this.

Logistic regression accuracy: 0.895
Always-pass baseline accuracy: 0.895
Confusion matrix (rows = actual [fail, pass], columns = predicted [fail, pass]):
[[ 3  1]
 [ 3 31]]


## Part 4 — Feature engineering

In [9]:
# Model 1: quiz average only
X_quiz = df[["quiz_avg"]]
Xq_train, Xq_test, yq_train, yq_test = train_test_split(
    X_quiz, df["score"], test_size=0.25, random_state=42
)
model_quiz = LinearRegression().fit(Xq_train, yq_train)
r2_quiz = r2_score(yq_test, model_quiz.predict(Xq_test))

# Model 2: all three numeric features
X_numeric = df[features]
Xn_train, Xn_test, yn_train, yn_test = train_test_split(
    X_numeric, df["score"], test_size=0.25, random_state=42
)
model_numeric = LinearRegression().fit(Xn_train, yn_train)
r2_numeric = r2_score(yn_test, model_numeric.predict(Xn_test))

# Model 3: all numeric features plus one-hot encoded program
X_engineered = pd.get_dummies(df[features + ["program"]], columns=["program"], dtype=int)
Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    X_engineered, df["score"], test_size=0.25, random_state=42
)
model_engineered = LinearRegression().fit(Xe_train, ye_train)
r2_engineered = r2_score(ye_test, model_engineered.predict(Xe_test))

print(f"Quiz only R²: {r2_quiz:.3f}")
print(f"Three numeric features R²: {r2_numeric:.3f}")
print(f"Numeric + program R²: {r2_engineered:.3f}")
# Adding assignments and attendance improves the quiz-only model substantially. Program indicators
# do not improve on the numeric-feature model here, showing that extra columns are not automatically useful.

Quiz only R²: 0.624
Three numeric features R²: 0.887
Numeric + program R²: 0.880


## Part 5 — Ship a dashboard

### Exercise 5.1 — The story in three charts

In [10]:
# 1. Distribution of assessment scores
fig1 = px.histogram(df, x="score", nbins=12, title="1. Where the class stands")
fig1.show()

In [11]:
# 2. Quiz performance compared with final score; color identifies pass/fail
fig2 = px.scatter(
    df, x="quiz_avg", y="score", color="passed", hover_name="name",
    title="2. Who is at risk"
)
fig2.show()

In [12]:
# 3. Average score for each district
district_average = df.groupby("district", as_index=False)["score"].mean()
fig3 = px.bar(
    district_average, x="district", y="score",
    title="3. Where to act"
)
fig3.show()

### Exercise 5.2 — Publish it

In [13]:
# Save the at-risk scatter plot as a standalone interactive HTML dashboard.
fig2.write_html("class_dashboard.html", include_plotlyjs=True, full_html=True)
print("Saved class_dashboard.html")

Saved class_dashboard.html


## Bonus — One chart, four stories

In [14]:
fig_bonus = px.box(
    df, x="program", y="score", color="program",
    title="Bonus: Score spread by program"
)
fig_bonus.show()
program_spread = df.groupby("program")["score"].agg(["std", "min", "max", "count"]).sort_values("std")
print(program_spread)
most_consistent = program_spread.index[0]
widest_spread = program_spread.index[-1]
print(f"{most_consistent} has the smallest standard deviation and is the most consistent in this dataset.")
print(f"{widest_spread} has the largest standard deviation and the widest spread in this dataset.")
# Consistency and spread are judged by standard deviation; the box plot also shows medians and quartiles.

                            std   min   max  count
program                                           
Networking            10.838729  41.9  89.4     37
Accounting            11.047231  43.0  89.5     37
Information Systems   12.262727  28.6  84.4     38
Software Engineering  13.822742  37.2  92.9     38
Networking has the smallest standard deviation and is the most consistent in this dataset.
Software Engineering has the largest standard deviation and the widest spread in this dataset.


## Part 6 — Model card

**Purpose:** Estimate a student's score from quiz average, assignment average, and attendance percentage to support early academic review.

**Data and features:** The model uses 150 student records. The three numeric input features are `quiz_avg`, `assignments_avg`, and `attendance_pct`; `score` is the target. The split contains 112 training records and 38 held-out test records.

**Performance:** On the held-out test set, the model's MAE is reported above in score points and its R² indicates the share of score variation explained on that test set. The typical absolute error is approximately the MAE number of points.

**Risks and use:** Estimates can be wrong and may reflect limitations in the historical sample. Use them only as a preliminary prompt for supportive human review. Do not use them as final grades, as a definitive measure of ability, or as the sole basis for decisions that affect students.